# Import Libraries and Data

In [5]:
import polars as pl
import warnings
import os 

warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)
os.environ['POLARS_WARN_UNSTABLE'] = '0'

In [6]:
# Configure Polars to show all rows and make it scrollable
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Set wider table width

polars.config.Config

In [7]:
# Load all data files using Polars
print("Loading datasets with Polars...")
pos_cash_df = pl.read_csv('../data/POS_CASH_balance.csv')

print(f"POS Cash balance shape: {pos_cash_df.shape}")

Loading datasets with Polars...
POS Cash balance shape: (10001358, 8)


# Define Utils Function

In [8]:
def track_record_timeline(df, sk_id_curr=None, sk_id_prev=None):
    """
    Track how a specific record behaves over time by querying SK_ID_* fields
    and sorting by MONTHS_BALANCE to show the timeline.
    
    Parameters:
    - df: POS_CASH_balance DataFrame
    - sk_id_curr: Customer ID to filter by (optional)
    - sk_id_prev: Previous loan ID to filter by (optional)
    
    Returns:
    - DataFrame sorted by MONTHS_BALANCE showing timeline behavior
    """
    
    # Start with the full dataset
    filtered_df = df.clone()
    
    # Apply filters based on provided IDs
    if sk_id_curr is not None:
        filtered_df = filtered_df.filter(pl.col('SK_ID_CURR') == sk_id_curr)
        print(f"Filtering by SK_ID_CURR: {sk_id_curr}")
    
    if sk_id_prev is not None:
        filtered_df = filtered_df.filter(pl.col('SK_ID_PREV') == sk_id_prev)
        print(f"Filtering by SK_ID_PREV: {sk_id_prev}")
    
    # Sort by MONTHS_BALANCE (most recent first: -1, -2, -3, etc.)
    timeline_df = filtered_df.sort('MONTHS_BALANCE', descending=True)
    
    print(f"Found {timeline_df.shape[0]} records")
    
    if timeline_df.shape[0] > 0:
        print(f"Timeline range: {timeline_df['MONTHS_BALANCE'].max()} to {timeline_df['MONTHS_BALANCE'].min()}")
    
    return timeline_df

track_record_timeline(pos_cash_df,sk_id_curr=182943)

Filtering by SK_ID_CURR: 182943
Found 43 records
Timeline range: -2 to -62


SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
i64,i64,i64,f64,f64,str,i64,i64
1803195,182943,-2,48.0,15.0,"""Active""",0,0
1803195,182943,-3,48.0,16.0,"""Active""",0,0
1803195,182943,-4,48.0,17.0,"""Active""",0,0
1803195,182943,-5,48.0,18.0,"""Active""",0,0
1803195,182943,-6,48.0,19.0,"""Active""",0,0
1803195,182943,-7,48.0,20.0,"""Active""",0,0
1803195,182943,-8,48.0,21.0,"""Active""",0,0
1803195,182943,-9,48.0,22.0,"""Active""",0,0
1803195,182943,-10,48.0,23.0,"""Active""",0,0


In [9]:
def analyze_loan_terms(df):
    """
    Analyze loan terms from POS_CASH_balance data using CNT_INSTALMENT field.
    Categorize loans as short-term, medium-term, or long-term.
    
    Parameters:
    - df: POS_CASH_balance DataFrame
    
    Returns:
    - DataFrame with loan term analysis and categories
    """
    
    print("=== LOAN TERM ANALYSIS ===")
    
    # Get the original loan terms (CNT_INSTALMENT represents total installments)
    loan_terms = df.select(['SK_ID_PREV', 'CNT_INSTALMENT']).drop_nulls().unique()
    
    print(f"Total unique loans: {loan_terms.shape[0]:,}")
    
    # Analyze installment distribution
    print(f"\nInstallment term statistics:")
    print(f"Min installments: {loan_terms['CNT_INSTALMENT'].min()}")
    print(f"Max installments: {loan_terms['CNT_INSTALMENT'].max()}")
    print(f"Mean installments: {loan_terms['CNT_INSTALMENT'].mean():.1f}")
    print(f"Median installments: {loan_terms['CNT_INSTALMENT'].median()}")
    
    # Create loan term categories
    # Assuming monthly installments: 
    # Short-term: <= 12 months
    # Medium-term: 13-24 months  
    # Long-term: > 24 months
    
    loan_categories = loan_terms.with_columns([
        pl.when(pl.col('CNT_INSTALMENT') <= 12)
        .then(pl.lit('Short-term (≤12 months)'))
        .when(pl.col('CNT_INSTALMENT') <= 24)
        .then(pl.lit('Medium-term (13-24 months)'))
        .otherwise(pl.lit('Long-term (>24 months)'))
        .alias('loan_term_category')
    ])
    
    # Distribution analysis
    term_distribution = loan_categories.group_by('loan_term_category').agg([
        pl.count().alias('loan_count'),
        pl.col('CNT_INSTALMENT').mean().alias('avg_installments'),
        pl.col('CNT_INSTALMENT').min().alias('min_installments'),
        pl.col('CNT_INSTALMENT').max().alias('max_installments')
    ]).sort('loan_count', descending=True)
    
    print(f"\nLoan term distribution:")
    print(term_distribution)
    
    # Most common installment counts
    print(f"\nMost common installment counts:")
    common_terms = loan_terms.group_by('CNT_INSTALMENT').agg(pl.count().alias('count')).sort('count', descending=True)
    print(common_terms.head(10))
    
    return loan_categories

loan_term_analysis_df = analyze_loan_terms(pos_cash_df)

=== LOAN TERM ANALYSIS ===
Total unique loans: 1,242,595

Installment term statistics:
Min installments: 1.0
Max installments: 92.0
Mean installments: 13.2
Median installments: 11.0

Loan term distribution:
shape: (3, 5)
┌────────────────────────────┬────────────┬──────────────────┬──────────────────┬──────────────────┐
│ loan_term_category         ┆ loan_count ┆ avg_installments ┆ min_installments ┆ max_installments │
│ ---                        ┆ ---        ┆ ---              ┆ ---              ┆ ---              │
│ str                        ┆ u32        ┆ f64              ┆ f64              ┆ f64              │
╞════════════════════════════╪════════════╪══════════════════╪══════════════════╪══════════════════╡
│ Short-term (≤12 months)    ┆ 916120     ┆ 8.321648         ┆ 1.0              ┆ 12.0             │
│ Medium-term (13-24 months) ┆ 223114     ┆ 20.422484        ┆ 13.0             ┆ 24.0             │
│ Long-term (>24 months)     ┆ 103361     ┆ 41.330008        ┆ 25.0     

In [16]:
def aggregate_pos_step1_loan_level_simple(df):
    """
    Step 1: Simple and working aggregation to loan level.
    
    Parameters:
    - df: POS_CASH_balance DataFrame
    
    Returns:
    - DataFrame with one row per SK_ID_PREV
    """
    
    print("=== STEP 1: SIMPLE AGGREGATION TO LOAN LEVEL (SK_ID_PREV) ===")
    
    # Basic aggregations first
    basic_agg = df.group_by(['SK_ID_PREV', 'SK_ID_CURR']).agg([
        pl.count().alias('TOTAL_MONTHS_OBSERVED'),
        
        # Basic DPD stats
        pl.col('SK_DPD').max().alias('MAX_DPD'),
        pl.col('SK_DPD').mean().alias('MEAN_DPD'),
        pl.col('SK_DPD_DEF').max().alias('MAX_DPD_DEF'),
        
        # Installment stats
        pl.col('CNT_INSTALMENT').max().alias('MAX_CNT_INSTALMENT'),
        pl.col('CNT_INSTALMENT_FUTURE').min().alias('MIN_CNT_INSTALMENT_FUTURE'),
        
        # Time range
        pl.col('MONTHS_BALANCE').min().alias('EARLIEST_MONTH'),
        pl.col('MONTHS_BALANCE').max().alias('LATEST_MONTH'),
    ])
    
    # Count DPD months using separate aggregations
    dpd_positive = df.filter(pl.col('SK_DPD') > 0).group_by(['SK_ID_PREV']).agg([
        pl.count().alias('MONTHS_DPD_POSITIVE')
    ])
    
    dpd_7plus = df.filter(pl.col('SK_DPD') > 7).group_by(['SK_ID_PREV']).agg([
        pl.count().alias('MONTHS_DPD_7PLUS')
    ])
    
    dpd_30plus = df.filter(pl.col('SK_DPD') > 30).group_by(['SK_ID_PREV']).agg([
        pl.count().alias('MONTHS_DPD_30PLUS')
    ])
    
    # Count specific statuses
    active_count = df.filter(pl.col('NAME_CONTRACT_STATUS') == 'Active').group_by(['SK_ID_PREV']).agg([
        pl.count().alias('MONTHS_ACTIVE')
    ])
    
    completed_count = df.filter(pl.col('NAME_CONTRACT_STATUS') == 'Completed').group_by(['SK_ID_PREV']).agg([
        pl.count().alias('MONTHS_COMPLETED')
    ])
    
    demand_count = df.filter(pl.col('NAME_CONTRACT_STATUS') == 'Demand').group_by(['SK_ID_PREV']).agg([
        pl.count().alias('MONTHS_DEMAND')
    ])
    
    # Join all counts
    loan_features = basic_agg
    loan_features = loan_features.join(dpd_positive, on='SK_ID_PREV', how='left')
    loan_features = loan_features.join(dpd_7plus, on='SK_ID_PREV', how='left')
    loan_features = loan_features.join(dpd_30plus, on='SK_ID_PREV', how='left')
    loan_features = loan_features.join(active_count, on='SK_ID_PREV', how='left')
    loan_features = loan_features.join(completed_count, on='SK_ID_PREV', how='left')
    loan_features = loan_features.join(demand_count, on='SK_ID_PREV', how='left')
    
    # Fill nulls with 0
    loan_features = loan_features.fill_null(0)
    
    # Calculate derived features
    loan_features = loan_features.with_columns([
        # Repayment progress
        pl.when(pl.col('MAX_CNT_INSTALMENT') > 0)
        .then(1.0 - (pl.col('MIN_CNT_INSTALMENT_FUTURE') / pl.col('MAX_CNT_INSTALMENT')))
        .otherwise(0.0).alias('REPAYMENT_PROGRESS_RATIO'),
        
        # DPD rates
        (pl.col('MONTHS_DPD_POSITIVE') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('DPD_RATE'),
        (pl.col('MONTHS_DPD_7PLUS') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('DPD_7PLUS_RATE'),
        (pl.col('MONTHS_DPD_30PLUS') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('DPD_30PLUS_RATE'),
        
        # Status rates
        (pl.col('MONTHS_ACTIVE') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('ACTIVE_RATE'),
        (pl.col('MONTHS_COMPLETED') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('COMPLETED_RATE'),
        (pl.col('MONTHS_DEMAND') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('DEMAND_RATE'),
        
        # Problem flags
        (pl.col('MONTHS_DEMAND') > 0).cast(pl.Int8).alias('EVER_DEMAND'),
        (pl.col('MAX_DPD') > 30).cast(pl.Int8).alias('SEVERE_DPD_FLAG'),
        (pl.col('MONTHS_DEMAND') > 0).cast(pl.Int8).alias('IS_PROBLEM_LOAN'),
    ])
    
    print(f"Step 1 complete: {df.shape[0]:,} monthly records → {loan_features.shape[0]:,} loan-level records")
    print(f"Features created: {loan_features.shape[1]} columns")
    
    return loan_features

In [19]:
def aggregate_pos_step2_customer_level_simple(loan_level_df):
    """
    Step 2: Simple customer-level aggregation that matches the simplified Step 1 output.
    Only references columns that exist in aggregate_pos_step1_loan_level_simple() output.
    
    Parameters:
    - loan_level_df: DataFrame from simple Step 1 function
    
    Returns:
    - DataFrame with one row per SK_ID_CURR with customer features
    """
    
    print("=== STEP 2: SIMPLE AGGREGATION TO CUSTOMER LEVEL (SK_ID_CURR) ===")
    
    customer_level_features = loan_level_df.group_by('SK_ID_CURR').agg([
        
        # === BASIC LOAN COUNTS ===
        pl.count().alias('POS_COUNT_LOANS_TOTAL'),
        pl.col('IS_PROBLEM_LOAN').sum().alias('POS_COUNT_PROBLEM_LOANS'),
        pl.col('EVER_DEMAND').sum().alias('POS_COUNT_LOANS_DEMAND'),
        
        # === DPD STATISTICS ACROSS LOANS ===
        pl.col('MAX_DPD').mean().alias('POS_MEAN_MAX_DPD'),
        pl.col('MAX_DPD').max().alias('POS_MAX_MAX_DPD'),
        pl.col('MAX_DPD').min().alias('POS_MIN_MAX_DPD'),
        
        pl.col('MEAN_DPD').mean().alias('POS_MEAN_MEAN_DPD'),
        pl.col('DPD_RATE').mean().alias('POS_MEAN_DPD_RATE'),
        pl.col('DPD_RATE').max().alias('POS_MAX_DPD_RATE'),
        
        # === SEVERE DPD STATISTICS ===
        pl.col('DPD_7PLUS_RATE').mean().alias('POS_MEAN_DPD_7PLUS_RATE'),
        pl.col('DPD_7PLUS_RATE').max().alias('POS_MAX_DPD_7PLUS_RATE'),
        pl.col('DPD_30PLUS_RATE').mean().alias('POS_MEAN_DPD_30PLUS_RATE'),
        pl.col('DPD_30PLUS_RATE').max().alias('POS_MAX_DPD_30PLUS_RATE'),
        
        # === REPAYMENT STATISTICS ===
        pl.col('REPAYMENT_PROGRESS_RATIO').mean().alias('POS_MEAN_REPAYMENT_PROGRESS'),
        pl.col('REPAYMENT_PROGRESS_RATIO').max().alias('POS_MAX_REPAYMENT_PROGRESS'),
        pl.col('REPAYMENT_PROGRESS_RATIO').min().alias('POS_MIN_REPAYMENT_PROGRESS'),
        
        # === CONTRACT STATISTICS ===
        pl.col('MAX_CNT_INSTALMENT').mean().alias('POS_MEAN_CONTRACT_LENGTH'),
        pl.col('MAX_CNT_INSTALMENT').max().alias('POS_MAX_CONTRACT_LENGTH'), 
        pl.col('MAX_CNT_INSTALMENT').min().alias('POS_MIN_CONTRACT_LENGTH'),
        
        # === STATUS RATE AVERAGES ===
        pl.col('COMPLETED_RATE').mean().alias('POS_MEAN_COMPLETED_RATE'),
        pl.col('ACTIVE_RATE').mean().alias('POS_MEAN_ACTIVE_RATE'),
        pl.col('DEMAND_RATE').mean().alias('POS_MEAN_DEMAND_RATE'),
        
        # === HIGH-RISK FLAGS (ANY LOAN EVER...) ===
        pl.col('EVER_DEMAND').max().cast(pl.Int8).alias('POS_FLAG_ANY_DEMAND'),
        pl.col('SEVERE_DPD_FLAG').max().cast(pl.Int8).alias('POS_FLAG_ANY_SEVERE_DPD'),
        
        # === TIME RANGE STATISTICS ===
        pl.col('TOTAL_MONTHS_OBSERVED').sum().alias('POS_TOTAL_MONTHS_OBSERVED'),
        pl.col('EARLIEST_MONTH').min().alias('POS_EARLIEST_MONTH'),
        pl.col('LATEST_MONTH').max().alias('POS_LATEST_MONTH'),
    ])
    
    # Calculate ratios and risk segments
    customer_level_features = customer_level_features.with_columns([
        # === LOAN RATIOS ===
        (pl.col('POS_COUNT_PROBLEM_LOANS') / pl.col('POS_COUNT_LOANS_TOTAL')).alias('POS_RATIO_PROBLEM_LOANS'),
        (pl.col('POS_COUNT_LOANS_DEMAND') / pl.col('POS_COUNT_LOANS_TOTAL')).alias('POS_RATIO_LOANS_DEMAND'),
        
        # === OBSERVATION TIME SPAN ===
        (pl.col('POS_LATEST_MONTH') - pl.col('POS_EARLIEST_MONTH')).alias('POS_OBSERVATION_SPAN_MONTHS'),
        
        # === SIMPLIFIED RISK SEGMENT ===
        pl.when(pl.col('POS_MAX_MAX_DPD') > 90)
        .then(pl.lit('HIGH_RISK'))
        .when((pl.col('POS_FLAG_ANY_DEMAND') == 1) | (pl.col('POS_MAX_MAX_DPD') > 30))
        .then(pl.lit('MEDIUM_RISK'))
        .when(pl.col('POS_MEAN_DPD_RATE') > 0.05)
        .then(pl.lit('LOW_MEDIUM_RISK'))
        .otherwise(pl.lit('LOW_RISK')).alias('POS_RISK_SEGMENT'),
    ])
    
    print(f"Step 2 complete: {loan_level_df.shape[0]:,} loan-level records → {customer_level_features.shape[0]:,} customer-level records")
    print(f"Features created: {customer_level_features.shape[1]} columns")
    
    return customer_level_features

In [20]:
loan_features = aggregate_pos_step1_loan_level_simple(pos_cash_df)
customer_features = aggregate_pos_step2_customer_level_simple(loan_features)
customer_features.head(5)

=== STEP 1: SIMPLE AGGREGATION TO LOAN LEVEL (SK_ID_PREV) ===
Step 1 complete: 10,001,358 monthly records → 936,325 loan-level records
Features created: 26 columns
=== STEP 2: SIMPLE AGGREGATION TO CUSTOMER LEVEL (SK_ID_CURR) ===
Step 2 complete: 936,325 loan-level records → 337,252 customer-level records
Features created: 32 columns


SK_ID_CURR,POS_COUNT_LOANS_TOTAL,POS_COUNT_PROBLEM_LOANS,POS_COUNT_LOANS_DEMAND,POS_MEAN_MAX_DPD,POS_MAX_MAX_DPD,POS_MIN_MAX_DPD,POS_MEAN_MEAN_DPD,POS_MEAN_DPD_RATE,POS_MAX_DPD_RATE,POS_MEAN_DPD_7PLUS_RATE,POS_MAX_DPD_7PLUS_RATE,POS_MEAN_DPD_30PLUS_RATE,POS_MAX_DPD_30PLUS_RATE,POS_MEAN_REPAYMENT_PROGRESS,POS_MAX_REPAYMENT_PROGRESS,POS_MIN_REPAYMENT_PROGRESS,POS_MEAN_CONTRACT_LENGTH,POS_MAX_CONTRACT_LENGTH,POS_MIN_CONTRACT_LENGTH,POS_MEAN_COMPLETED_RATE,POS_MEAN_ACTIVE_RATE,POS_MEAN_DEMAND_RATE,POS_FLAG_ANY_DEMAND,POS_FLAG_ANY_SEVERE_DPD,POS_TOTAL_MONTHS_OBSERVED,POS_EARLIEST_MONTH,POS_LATEST_MONTH,POS_RATIO_PROBLEM_LOANS,POS_RATIO_LOANS_DEMAND,POS_OBSERVATION_SPAN_MONTHS,POS_RISK_SEGMENT
i64,u32,i64,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,u32,i64,i64,f64,f64,i64,str
267414,2,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.666667,1.0,0.333333,18.0,24.0,12.0,0.045455,0.954545,0.0,0,0,20,-14,-1,0.0,0.0,13,"""LOW_RISK"""
129256,1,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,16.0,16.0,16.0,0.066667,0.933333,0.0,0,0,15,-29,-15,0.0,0.0,14,"""LOW_RISK"""
341081,4,0,0,1.5,6,0,0.115385,0.019231,0.076923,0.0,0.0,0.0,0.0,1.0,1.0,1.0,12.0,12.0,12.0,0.076923,0.923077,0.0,0,0,52,-71,-3,0.0,0.0,68,"""LOW_RISK"""
369788,1,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,10.0,10.0,10.0,0.090909,0.909091,0.0,0,0,11,-91,-81,0.0,0.0,10,"""LOW_RISK"""
198722,1,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,10.0,10.0,10.0,0.090909,0.909091,0.0,0,0,11,-26,-16,0.0,0.0,10,"""LOW_RISK"""


# EDA

In [42]:
pos_cash_df.sort('SK_ID_CURR').head()

SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
i64,i64,i64,f64,f64,str,i64,i64
1851984,100001,-96,4.0,2.0,"""Active""",0,0
1851984,100001,-95,4.0,1.0,"""Active""",7,7
1369693,100001,-53,4.0,0.0,"""Completed""",0,0
1369693,100001,-54,4.0,1.0,"""Active""",0,0
1851984,100001,-93,4.0,0.0,"""Completed""",0,0


In [37]:
print(f"\nColumn info:")                                                         
for col in pos_cash_df.columns:                                                  
    dtype = pos_cash_df.schema[col]                                              
    missing_count = pos_cash_df[col].null_count()                                
    missing_pct = missing_count / pos_cash_df.shape[0] * 100                     
    print(f"  {col:25s} | {str(dtype):15s} | Missing: {missing_count:6,}({missing_pct:5.1f}%)")     


Column info:
  SK_ID_PREV                | Int64           | Missing:      0(  0.0%)
  SK_ID_CURR                | Int64           | Missing:      0(  0.0%)
  MONTHS_BALANCE            | Int64           | Missing:      0(  0.0%)
  CNT_INSTALMENT            | Float64         | Missing: 26,071(  0.3%)
  CNT_INSTALMENT_FUTURE     | Float64         | Missing: 26,087(  0.3%)
  NAME_CONTRACT_STATUS      | String          | Missing:      0(  0.0%)
  SK_DPD                    | Int64           | Missing:      0(  0.0%)
  SK_DPD_DEF                | Int64           | Missing:      0(  0.0%)


In [38]:
# IMPORTANT: Time Analysis - Understanding MONTHS_BALANCE
print("=== TIME INFORMATION ANALYSIS ===")
print("You're right - let's understand how time works in this dataset!")

print("\n1. MONTHS_BALANCE field analysis:")
print("From the data dictionary: 'Month of balance relative to application date'")
print("(-1 means the freshest balance date)")

months_analysis = pos_cash_df.group_by('MONTHS_BALANCE').agg([
    pl.count().alias('records'),
    pl.col('SK_ID_CURR').n_unique().alias('unique_customers')
]).sort('MONTHS_BALANCE', descending=True)

print(f"\nMONTHS_BALANCE distribution (most recent first):")
print(months_analysis.head(20))

print(f"\nRange: {pos_cash_df['MONTHS_BALANCE'].min()} to {pos_cash_df['MONTHS_BALANCE'].max()}")
print(f"This means we have data from {abs(pos_cash_df['MONTHS_BALANCE'].min())} months ago to {abs(pos_cash_df['MONTHS_BALANCE'].max())} months ago")

# Let's see the relationship between months and contract status
print(f"\n2. Contract status by time period:")
recent_status = pos_cash_df.filter(pl.col('MONTHS_BALANCE') >= -6).group_by('NAME_CONTRACT_STATUS').agg(pl.count().alias('recent_count'))
older_status = pos_cash_df.filter(pl.col('MONTHS_BALANCE') < -24).group_by('NAME_CONTRACT_STATUS').agg(pl.count().alias('older_count'))

status_comparison = recent_status.join(older_status, on='NAME_CONTRACT_STATUS', how='outer').fill_null(0)
print("Recent (last 6 months) vs Older (24+ months ago):")
print(status_comparison)

=== TIME INFORMATION ANALYSIS ===
You're right - let's understand how time works in this dataset!

1. MONTHS_BALANCE field analysis:
From the data dictionary: 'Month of balance relative to application date'
(-1 means the freshest balance date)

MONTHS_BALANCE distribution (most recent first):
shape: (20, 3)
┌────────────────┬─────────┬──────────────────┐
│ MONTHS_BALANCE ┆ records ┆ unique_customers │
│ ---            ┆ ---     ┆ ---              │
│ i64            ┆ u32     ┆ u32              │
╞════════════════╪═════════╪══════════════════╡
│ -1             ┆ 94908   ┆ 79744            │
│ -2             ┆ 169529  ┆ 144653           │
│ -3             ┆ 183589  ┆ 155746           │
│ -4             ┆ 193147  ┆ 162953           │
│ -5             ┆ 200726  ┆ 168437           │
│ -6             ┆ 206849  ┆ 172590           │
│ -7             ┆ 210229  ┆ 175140           │
│ -8             ┆ 214149  ┆ 178121           │
│ -9             ┆ 215558  ┆ 179328           │
│ -10            ┆ 

In [39]:
# Payment Discipline Analysis
print("=== PAYMENT DISCIPLINE ANALYSIS ===")

# 1. DPD (Days Past Due) Distribution
print("\n1. DPD Distribution:")
dpd_dist = pos_cash_df.group_by('SK_DPD').agg(pl.count().alias('count')).sort('SK_DPD')
print("SK_DPD distribution:")
print(dpd_dist.head(20))  # Show first 20 DPD values

print(f"\nDPD Summary Statistics:")
print(f"Max DPD: {pos_cash_df['SK_DPD'].max()}")
print(f"Mean DPD: {pos_cash_df['SK_DPD'].mean():.2f}")
print(f"Records with DPD > 0: {pos_cash_df.filter(pl.col('SK_DPD') > 0).shape[0]:,} ({pos_cash_df.filter(pl.col('SK_DPD') > 0).shape[0]/pos_cash_df.shape[0]*100:.1f}%)")

=== PAYMENT DISCIPLINE ANALYSIS ===

1. DPD Distribution:
SK_DPD distribution:
shape: (20, 2)
┌────────┬─────────┐
│ SK_DPD ┆ count   │
│ ---    ┆ ---     │
│ i64    ┆ u32     │
╞════════╪═════════╡
│ 0      ┆ 9706131 │
│ 1      ┆ 21872   │
│ 2      ┆ 17358   │
│ 3      ┆ 14403   │
│ 4      ┆ 12350   │
│ 5      ┆ 11046   │
│ 6      ┆ 9615    │
│ 7      ┆ 8332    │
│ 8      ┆ 7360    │
│ 9      ┆ 6668    │
│ 10     ┆ 6049    │
│ 11     ┆ 5392    │
│ 12     ┆ 4907    │
│ 13     ┆ 4468    │
│ 14     ┆ 3944    │
│ 15     ┆ 3438    │
│ 16     ┆ 3169    │
│ 17     ┆ 2980    │
│ 18     ┆ 2759    │
│ 19     ┆ 2532    │
└────────┴─────────┘

DPD Summary Statistics:
Max DPD: 4231
Mean DPD: 11.61
Records with DPD > 0: 295,227 (3.0%)


In [40]:
# 2. Contract Status Analysis
print("\n2. Contract Status Analysis:")
status_dist = pos_cash_df.group_by('NAME_CONTRACT_STATUS').agg([
    pl.count().alias('count'),
    pl.col('SK_DPD').mean().alias('avg_dpd'),
    pl.col('SK_DPD').max().alias('max_dpd')
]).sort('count', descending=True)
print(status_dist)

# 3. Time-based Analysis (MONTHS_BALANCE)
print("\n3. Time Analysis:")
print(f"MONTHS_BALANCE range: {pos_cash_df['MONTHS_BALANCE'].min()} to {pos_cash_df['MONTHS_BALANCE'].max()}")
print("Recent months analysis (last 6 months):")
recent_months = pos_cash_df.filter(pl.col('MONTHS_BALANCE') >= -6)
print(f"Records in last 6 months: {recent_months.shape[0]:,}")
print(f"DPD rate in recent months: {recent_months.filter(pl.col('SK_DPD') > 0).shape[0]/recent_months.shape[0]*100:.1f}%")


2. Contract Status Analysis:
shape: (9, 4)
┌───────────────────────┬─────────┬─────────────┬─────────┐
│ NAME_CONTRACT_STATUS  ┆ count   ┆ avg_dpd     ┆ max_dpd │
│ ---                   ┆ ---     ┆ ---         ┆ ---     │
│ str                   ┆ u32     ┆ f64         ┆ i64     │
╞═══════════════════════╪═════════╪═════════════╪═════════╡
│ Active                ┆ 9151119 ┆ 9.889115    ┆ 4231    │
│ Completed             ┆ 744883  ┆ 24.803362   ┆ 3988    │
│ Signed                ┆ 87260   ┆ 0.022427    ┆ 233     │
│ Demand                ┆ 7065    ┆ 847.685492  ┆ 4050    │
│ Returned to the store ┆ 5461    ┆ 0.005493    ┆ 22      │
│ Approved              ┆ 4917    ┆ 0.00061     ┆ 3       │
│ Amortized debt        ┆ 636     ┆ 1764.290881 ┆ 3468    │
│ Canceled              ┆ 15      ┆ 0.0         ┆ 0       │
│ XNA                   ┆ 2       ┆ 0.0         ┆ 0       │
└───────────────────────┴─────────┴─────────────┴─────────┘

3. Time Analysis:
MONTHS_BALANCE range: -96 to -1
Recen

In [41]:
# 4. Customer-level Payment Discipline Analysis (FIXED v2)
print("\n4. Customer-Level Analysis:")

# Step-by-step approach to avoid complex aggregation errors
print("Calculating customer payment discipline metrics...")

# Basic customer stats
customer_basic = pos_cash_df.group_by('SK_ID_CURR').agg([
    pl.count().alias('total_records'),
    pl.col('SK_DPD').max().alias('max_dpd_ever'),
    pl.col('SK_DPD').mean().alias('avg_dpd')
])

# Count late payments per customer  
late_payments = pos_cash_df.filter(pl.col('SK_DPD') > 0).group_by('SK_ID_CURR').agg([
    pl.count().alias('late_payments')
])

# Join basic stats with late payment counts
customer_discipline = customer_basic.join(late_payments, on='SK_ID_CURR', how='left')
customer_discipline = customer_discipline.with_columns([
    pl.col('late_payments').fill_null(0)
])

# Calculate late payment rate
customer_discipline = customer_discipline.with_columns([
    (pl.col('late_payments') / pl.col('total_records')).alias('late_payment_rate')
])

print(f"Customer discipline summary:")
print(f"Total customers: {customer_discipline.shape[0]:,}")
print(f"Customers with any late payments: {customer_discipline.filter(pl.col('late_payments') > 0).shape[0]:,}")

# Show sample results
print("\nSample customer discipline metrics:")
print(customer_discipline.head(10))

# Simple distribution analysis without complex when/then
print("\nPayment discipline distribution:")
print(f"Perfect customers (0% late): {customer_discipline.filter(pl.col('late_payment_rate') == 0).shape[0]:,}")
print(f"Good customers (1-10% late): {customer_discipline.filter((pl.col('late_payment_rate') > 0) & (pl.col('late_payment_rate') <= 0.1)).shape[0]:,}")
print(f"Fair customers (11-30% late): {customer_discipline.filter((pl.col('late_payment_rate') > 0.1) & (pl.col('late_payment_rate') <= 0.3)).shape[0]:,}")
print(f"Poor customers (31-50% late): {customer_discipline.filter((pl.col('late_payment_rate') > 0.3) & (pl.col('late_payment_rate') <= 0.5)).shape[0]:,}")
print(f"Very poor customers (50%+ late): {customer_discipline.filter(pl.col('late_payment_rate') > 0.5).shape[0]:,}")

# Show top disciplined vs worst disciplined customers
print("\nTop 5 most disciplined customers:")
print(customer_discipline.filter(pl.col('total_records') >= 5).sort(['late_payment_rate', 'max_dpd_ever']).head())

print("\nWorst 5 disciplined customers:")
print(customer_discipline.filter(pl.col('total_records') >= 5).sort(['late_payment_rate', 'max_dpd_ever'], descending=[True, True]).head())


4. Customer-Level Analysis:
Calculating customer payment discipline metrics...
Customer discipline summary:
Total customers: 337,252
Customers with any late payments: 62,984

Sample customer discipline metrics:
shape: (10, 6)
┌────────────┬───────────────┬──────────────┬──────────┬───────────────┬───────────────────┐
│ SK_ID_CURR ┆ total_records ┆ max_dpd_ever ┆ avg_dpd  ┆ late_payments ┆ late_payment_rate │
│ ---        ┆ ---           ┆ ---          ┆ ---      ┆ ---           ┆ ---               │
│ i64        ┆ u32           ┆ i64          ┆ f64      ┆ u32           ┆ f64               │
╞════════════╪═══════════════╪══════════════╪══════════╪═══════════════╪═══════════════════╡
│ 174341     ┆ 11            ┆ 0            ┆ 0.0      ┆ 0             ┆ 0.0               │
│ 295463     ┆ 23            ┆ 0            ┆ 0.0      ┆ 0             ┆ 0.0               │
│ 220093     ┆ 13            ┆ 0            ┆ 0.0      ┆ 0             ┆ 0.0               │
│ 157308     ┆ 20            

In [21]:
# Test the pipeline and check for null values
print("=== TESTING PIPELINE AND NULL VALUE ANALYSIS ===")

# Step 1: Loan-level aggregation
loan_features_simple = aggregate_pos_step1_loan_level_simple(pos_cash_df)

# Step 2: Customer-level aggregation  
customer_features_simple = aggregate_pos_step2_customer_level_simple(loan_features_simple)

print(f"\n=== NULL VALUE CHECK ===")
print(f"Customer features shape: {customer_features_simple.shape}")

# Check for null values in each column
null_counts = []
for col in customer_features_simple.columns:
    null_count = customer_features_simple[col].null_count()
    null_pct = null_count / customer_features_simple.shape[0] * 100
    null_counts.append({
        'column': col,
        'null_count': null_count, 
        'null_percentage': round(null_pct, 2)
    })

# Convert to DataFrame for better display
import polars as pl
null_analysis = pl.DataFrame(null_counts)
print(f"\nNull value analysis:")
print(null_analysis.sort('null_count', descending=True))

=== TESTING PIPELINE AND NULL VALUE ANALYSIS ===
=== STEP 1: SIMPLE AGGREGATION TO LOAN LEVEL (SK_ID_PREV) ===
Step 1 complete: 10,001,358 monthly records → 936,325 loan-level records
Features created: 26 columns
=== STEP 2: SIMPLE AGGREGATION TO CUSTOMER LEVEL (SK_ID_CURR) ===
Step 2 complete: 936,325 loan-level records → 337,252 customer-level records
Features created: 32 columns

=== NULL VALUE CHECK ===
Customer features shape: (337252, 32)

Null value analysis:
shape: (32, 3)
┌─────────────────────────────┬────────────┬─────────────────┐
│ column                      ┆ null_count ┆ null_percentage │
│ ---                         ┆ ---        ┆ ---             │
│ str                         ┆ i64        ┆ f64             │
╞═════════════════════════════╪════════════╪═════════════════╡
│ SK_ID_CURR                  ┆ 0          ┆ 0.0             │
│ POS_COUNT_LOANS_TOTAL       ┆ 0          ┆ 0.0             │
│ POS_COUNT_PROBLEM_LOANS     ┆ 0          ┆ 0.0             │
│ POS_COUN

In [22]:
# Export customer features to CSV
print("=== EXPORTING CUSTOMER FEATURES TO CSV ===")

# Run the complete pipeline first
loan_features_simple = aggregate_pos_step1_loan_level_simple(pos_cash_df)
customer_features_simple = aggregate_pos_step2_customer_level_simple(loan_features_simple)

# Export to CSV
output_path = "../data/POS_CASH_balance_processing.csv"
customer_features_simple.write_csv(output_path)

print(f"Customer features exported to: {output_path}")
print(f"Shape: {customer_features_simple.shape}")
print(f"Features: {customer_features_simple.shape[1]} columns")
print(f"Customers: {customer_features_simple.shape[0]:,} unique customers")

# Show quick summary
print(f"\nColumns exported:")
for i, col in enumerate(customer_features_simple.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nFile ready for joining with baseline_data_modeling!")

=== EXPORTING CUSTOMER FEATURES TO CSV ===
=== STEP 1: SIMPLE AGGREGATION TO LOAN LEVEL (SK_ID_PREV) ===
Step 1 complete: 10,001,358 monthly records → 936,325 loan-level records
Features created: 26 columns
=== STEP 2: SIMPLE AGGREGATION TO CUSTOMER LEVEL (SK_ID_CURR) ===
Step 2 complete: 936,325 loan-level records → 337,252 customer-level records
Features created: 32 columns
Customer features exported to: ../data/POS_CASH_balance_processing.csv
Shape: (337252, 32)
Features: 32 columns
Customers: 337,252 unique customers

Columns exported:
   1. SK_ID_CURR
   2. POS_COUNT_LOANS_TOTAL
   3. POS_COUNT_PROBLEM_LOANS
   4. POS_COUNT_LOANS_DEMAND
   5. POS_MEAN_MAX_DPD
   6. POS_MAX_MAX_DPD
   7. POS_MIN_MAX_DPD
   8. POS_MEAN_MEAN_DPD
   9. POS_MEAN_DPD_RATE
  10. POS_MAX_DPD_RATE
  11. POS_MEAN_DPD_7PLUS_RATE
  12. POS_MAX_DPD_7PLUS_RATE
  13. POS_MEAN_DPD_30PLUS_RATE
  14. POS_MAX_DPD_30PLUS_RATE
  15. POS_MEAN_REPAYMENT_PROGRESS
  16. POS_MAX_REPAYMENT_PROGRESS
  17. POS_MIN_REPAYMENT_P